# Preprocessing for Modeling

This notebook prepares the cleaned dataset for modeling. It loads the cleaned CSV, separates features and target, encodes categorical variables, performs a stratified train/test split, scales numeric inputs, and saves the resulting arrays for downstream notebooks.


## 0. Load Cleaned Data

Load the cleaned dataset produced by the EDA step. The file already has ID columns removed and standardized text fields.


In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

PROCESSED_PATH  = "../data/processed/diabetes_clean.csv"
df = pd.read_csv(PROCESSED_PATH)

# Display the data
print(df.shape)
print(df["CLASS"].value_counts())
df.head()


(1000, 12)
CLASS
Y    844
N    103
P     53
Name: count, dtype: int64


,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
0,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,M,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N
2,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
3,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
4,M,33,7.1,46,4.9,4.9,1.0,0.8,2.0,0.4,21.0,N


## 1. Dataset Overview

The cleaned dataset has **1,000 rows** and **12 columns** (including the target). Target class counts are:

- `Y`: 844
- `N`: 103
- `P`: 53

These counts match the EDA output and are used as the baseline before splitting.


## 2. Feature / Target Split

Separate `CLASS` into `y` and keep the remaining columns as features `X`. Any ID-like columns (if present) are dropped from the feature set.


In [2]:
y = df["CLASS"]
X = df.drop(columns=["CLASS"])

# Drop ID-like columns if present
id_cols = [c for c in ["ID", "No_Pation"] if c in X.columns]
X = X.drop(columns=id_cols)
print("Dropped ID columns:", id_cols)


Dropped ID columns: []


## 3. Encode Categorical Features

Convert `Gender` into a numeric feature. This mapping uses `M -> 0` and `F -> 1` after standardizing text casing. If the column is missing, the step is skipped.


In [3]:
if "Gender" in X.columns:
    X["Gender"] = X["Gender"].astype(str).str.upper().str.strip()
    X["Gender"] = X["Gender"].map({"M": 0, "F": 1})

## 4. Stratified Train/Test Split and Scaling

Split the dataset into training and test sets using a **stratified split** (preserves class proportions) with `test_size=0.2` and `random_state=42`. This yields **800 training** rows and **200 test** rows.

Class distribution after the split:

- Train: Y=675, N=82, P=43
- Test: Y=169, N=21, P=10

Standard scaling is fit on the training data and applied to both train and test sets to keep feature scaling consistent.


In [4]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## 5. Save Preprocessed Arrays

Save the scaled features and target arrays to a `.npz` file for easy loading in modeling notebooks.


In [5]:
import numpy as np

np.savez(
    "../data/processed/split_scaled.npz",
    X_train_s=X_train_s,
    X_test_s=X_test_s,
    y_train=y_train.to_numpy(),
    y_test=y_test.to_numpy(),
)
print("Saved split to data/processed/split_scaled.npz")


Saved split to data/processed/split_scaled.npz
